# VisionServeX — Clean Task-Separated Benchmark (v34)

**Version:** VisionServeX v2.30.0
**Notebook:** v34 (clean rewrite — no v20/v2.16 legacy)
**Author:** Arash Sajjadi
**Date:** 2026-05-18

## Protocol

This notebook is the **scientific** benchmark notebook for VisionServeX.
It deliberately separates tasks so that comparable models are compared
against each other, and no smoke result is presented as a benchmark.

### Rules

1. Smoke ≠ benchmark.
2. Each task has its own leaderboard.
3. No "overall winner" across tasks.
4. Promptable segmentation is **not** compared against automatic segmentation.
5. Domain models (medical/agriculture/aerial) are smoke/demo only unless a
   real GT dataset is supplied by the user.
6. Every model ends in exactly one of the v2.30 final states:
   `smoke_passed`, `benchmark_passed`, `expected_blocker`,
   `dependency_required`, `download_failed_retryable`,
   `manual_checkpoint_required`, `license_blocked`, `dataset_required`,
   `upstream_unavailable`, `not_applicable`.
7. Display never shows raw `NaN`, `NOT_WIRED`, `v20:`, `v2.16`, or
   `failed_runtime` for parseable blockers.

### Inputs (already produced by v2.30 release)

- `reports/core_smoke_matrix_v34.json` (65 core models, v2.30 smoke,
  6 blockers reclassified as `dependency_required` /
  `download_failed_retryable`)
- `reports/libreyolo_doctor_v34.json`,
  `libreyolo_model_discovery_v34.json`,
  `libreyolo_license_audit_v34.json`
- `reports/detection_leaderboard_400_v227_source.csv` (real COCO val2017
  400-image leaderboard from v2.27)
- `reports/segmentation_auto_instance_400_v227_source.json`
- `reports/rfdetr_seg_schema_probe_v229.json`
- `reports/deimv2_hf_audit_v230.json`,
  `rtdetrv4_checkpoint_audit_v230.json`

### Outputs

All artifacts land in
`notebook/visionservex_v34_run/visionservex_v34_outputs/{reports,plots,visuals}/`.


## 1. Environment, imports, output directory

In [1]:
# v34 — environment + paths + shared helpers
from __future__ import annotations

import json
import os
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Display safety: forbid raw NaN/NOT_WIRED/etc. in any rendered table.
from visionservex.reporting import (
    is_nullish, render_nullable, render_table_for_notebook,
    NOT_APPLICABLE, NOT_COLLECTED, NOT_FOUND, NOT_RUN, NOT_APPLICABLE_SMOKE,
)

REPO_ROOT = Path(os.environ.get("VISIONSERVEX_REPO", "/home/arash/PycharmProjects/VisionServeX"))
os.chdir(REPO_ROOT)

OUT_ROOT = REPO_ROOT / "notebook/visionservex_v34_run/visionservex_v34_outputs"
REPORTS_DIR = OUT_ROOT / "reports"
PLOTS_DIR = OUT_ROOT / "plots"
VISUALS_DIR = OUT_ROOT / "visuals"
for d in (REPORTS_DIR, PLOTS_DIR, VISUALS_DIR):
    d.mkdir(parents=True, exist_ok=True)

SMOKE_ASSETS = REPO_ROOT / "tests/assets/smoke"
SOURCE_REPORTS = REPO_ROOT / "reports"

import visionservex
VSX_VERSION = visionservex.__version__
NOTEBOOK_VERSION = "v34"
print(f"VisionServeX {VSX_VERSION} - notebook {NOTEBOOK_VERSION}")
print(f"Python {sys.version.split()[0]}  /  Platform {platform.platform()}")
print(f"Output -> {OUT_ROOT}")


ImportError: cannot import name 'is_nullish' from 'visionservex.reporting' (/home/arash/PycharmProjects/VisionServeX/notebook/.venv/lib/python3.13/site-packages/visionservex/reporting/__init__.py)

In [ ]:
# Environment report
env = {
    "visionservex_version": VSX_VERSION,
    "notebook_version": NOTEBOOK_VERSION,
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "machine": platform.machine(),
}
try:
    import torch
    env["torch_version"] = torch.__version__
    env["cuda_available"] = bool(torch.cuda.is_available())
    if torch.cuda.is_available():
        env["gpu_name"] = torch.cuda.get_device_name(0)
        env["compute_capability"] = list(torch.cuda.get_device_capability(0))
        env["cuda_version"] = torch.version.cuda
except Exception as exc:
    env["torch_error"] = str(exc)[:200]

(REPORTS_DIR / "environment_report_v34.json").write_text(json.dumps(env, indent=2))
display_env = pd.DataFrame({"key": list(env.keys()), "value": [str(v) for v in env.values()]})
display_env


## 2. GPU report

In [ ]:
gpu = {"cuda_available": False}
try:
    import torch
    gpu["cuda_available"] = torch.cuda.is_available()
    if torch.cuda.is_available():
        gpu["gpu_name"] = torch.cuda.get_device_name(0)
        gpu["compute_capability"] = list(torch.cuda.get_device_capability(0))
        gpu["cuda_version"] = torch.version.cuda
        free, total = torch.cuda.mem_get_info()
        gpu["total_vram_gb"] = round(total / 1024**3, 2)
        gpu["free_vram_gb"] = round(free / 1024**3, 2)
        gpu["torch_version"] = torch.__version__
except Exception as exc:
    gpu["error"] = str(exc)[:200]
(REPORTS_DIR / "gpu_report_v34.json").write_text(json.dumps(gpu, indent=2))
pd.DataFrame({"key": list(gpu.keys()), "value": [str(v) for v in gpu.values()]})


## 3. Clean display helper

`clean_display_value(x, status=None)` is the only function used to render
metric cells in this notebook. It guarantees:

- `None` / `NaN` / `inf` → `not collected` / `not applicable` / `not found`
- `NOT_WIRED` / `failed_runtime` never appears.
- Numeric values are formatted with 4 decimals.

In [ ]:
def clean_display_value(x, *, status: str | None = None, label: str | None = None) -> str:
    return render_nullable(x, status=status, label=label)

ALLOWED_FINAL_STATES = (
    "smoke_passed", "benchmark_passed", "expected_blocker",
    "dependency_required", "download_failed_retryable",
    "manual_checkpoint_required", "license_blocked",
    "dataset_required", "upstream_unavailable", "not_applicable",
)

FORBIDDEN_DISPLAY_STRINGS = ("NOT_WIRED", "v20:", "v2.16", "UNAVAILABLE_OR_FAILED")


## 4. Smoke matrix summary (v2.30 reclassified for v34)

Source: `reports/core_smoke_matrix_v34.json`. Already reclassified so the
six v2.30 blockers carry precise codes (`DOWNLOAD_FAILED_RETRYABLE`,
`NATTEN_REQUIRED`, `FLORENCE2_TRANSFORMERS_VERSION_REQUIRED`).

In [ ]:
smoke = json.loads((SOURCE_REPORTS / "core_smoke_matrix_v34.json").read_text())
smoke_summary = smoke.get("summary", {})
n_smoke = smoke_summary.get("smoke_passed", 0)
n_blocker = sum(
    smoke_summary.get(k, 0)
    for k in ("expected_blocker", "dependency_required", "download_failed_retryable",
              "manual_checkpoint_required", "license_blocked")
)
print(f"Total advertised core models : {smoke_summary.get('total', 0)}")
print(f"smoke_passed                 : {n_smoke}")
print(f"dependency_required          : {smoke_summary.get('dependency_required', 0)}")
print(f"download_failed_retryable    : {smoke_summary.get('download_failed_retryable', 0)}")
print(f"expected_blocker             : {smoke_summary.get('expected_blocker', 0)}")
print(f"license_blocked              : {smoke_summary.get('license_blocked', 0)}")
print(f"manual_checkpoint_required   : {smoke_summary.get('manual_checkpoint_required', 0)}")
print(f"failed_runtime               : {smoke_summary.get('failed_runtime', 0)}")
print(f"unclassified                 : {smoke_summary.get('unclassified', 0)}")
print(f"package_bug_remaining        : {smoke_summary.get('package_bug_remaining', 0)}")

# Persist the v34 matrix into the notebook's output tree
(REPORTS_DIR / "model_smoke_matrix_v34.json").write_text(json.dumps(smoke, indent=2))
# CSV (clean — no NaN)
import csv as _csv
rows = smoke.get("rows", [])
fields = list(rows[0].keys()) if rows else []
with open(REPORTS_DIR / "model_smoke_matrix_v34.csv", "w", newline="") as fh:
    w = _csv.DictWriter(fh, fieldnames=fields)
    w.writeheader()
    for r in rows:
        w.writerow(r)
print(f"Wrote -> {REPORTS_DIR / 'model_smoke_matrix_v34.csv'}")


In [ ]:
# Smoke state counts plot
state_counts = {}
for r in smoke["rows"]:
    state_counts[r["final_state"]] = state_counts.get(r["final_state"], 0) + 1
sc = pd.Series(state_counts).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 4))
sc.plot(kind="barh", ax=ax, color=["#76b041" if k == "smoke_passed" else "#d97700" for k in sc.index])
ax.set_xlabel("Number of models")
ax.set_title("v34 — smoke matrix final-state counts (core)")
for i, v in enumerate(sc.values):
    ax.text(v + 0.3, i, str(int(v)), va="center")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "model_smoke_state_counts.png", dpi=130)
plt.show()


## 5. LibreYOLO license audit + model inclusion policy

In [ ]:
ly_doctor = json.loads((SOURCE_REPORTS / "libreyolo_doctor_v34.json").read_text())
ly_audit = json.loads((SOURCE_REPORTS / "libreyolo_license_audit_v34.json").read_text())
ly_models = json.loads((SOURCE_REPORTS / "libreyolo_model_discovery_v34.json").read_text())

print(f"LibreYOLO installed : {ly_doctor.get('libreyolo_installed')}")
print(f"LibreYOLO version   : {ly_doctor.get('libreyolo_version')}")
print(f"n_weights discovered: {ly_models.get('n_weights')}")
print()
families_df = pd.DataFrame(ly_audit["rows"])
families_df_display = families_df[["family","code_license","weight_license","license_risk","auto_pull"]].copy()
families_df_display


In [ ]:
# Default-safe filter (Apache-2.0 / MIT only)
weights = ly_models.get("weights", [])
default_safe = [w for w in weights if w.get("license_risk") == "none" and any(
    ok in (w.get("weight_license") or "").upper() for ok in ("APACHE", "MIT")
)]
blocked = [w for w in weights if w not in default_safe]

n_safe = len(default_safe)
n_blocked = len(blocked)
print(f"LibreYOLO weights total       : {len(weights)}")
print(f"  default-safe (auto-pull)    : {n_safe}")
print(f"  blocked / opt-in            : {n_blocked}")

# License audit table to disk
import csv as _csv
fields = ["model_id","family","task","weight_license","license_risk","auto_pull","url"]
with open(REPORTS_DIR / "libreyolo_license_audit_v34.csv", "w", newline="") as fh:
    w = _csv.DictWriter(fh, fieldnames=fields, extrasaction="ignore")
    w.writeheader()
    for r in weights:
        w.writerow(r)
(REPORTS_DIR / "libreyolo_license_audit_v34.json").write_text(json.dumps(ly_audit, indent=2))

# License risk breakdown plot
risk_counts = pd.Series([w.get("license_risk", "unknown") for w in weights]).value_counts()
fig, ax = plt.subplots(figsize=(6,4))
risk_counts.plot(kind="bar", ax=ax,
                 color=["#76b041" if k == "none" else "#d97700" for k in risk_counts.index])
ax.set_title("LibreYOLO weights by license risk")
ax.set_ylabel("number of weights")
for i, v in enumerate(risk_counts.values):
    ax.text(i, v + 0.5, str(int(v)), ha="center")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "libre_yolo_license_breakdown.png", dpi=130)
plt.show()


## 6. Classification — smoke only (no labelled dataset supplied)

The notebook does **not** claim ImageNet accuracy because no labelled
ImageNet validation set is staged. We display the per-model smoke status
plus a synthesised top-k example.

In [ ]:
CLASSIFY_MODELS = [
    "swinv2-tiny", "convnextv2-tiny", "convnextv2-base",
    "maxvit-tiny-tf-224", "swinv2-small", "swinv2-base", "swinv2-large",
]
rows = []
for m in smoke["rows"]:
    if m["model_id"] in CLASSIFY_MODELS:
        rows.append({
            "model_id": m["model_id"],
            "task": m["task"],
            "final_state": m["final_state"],
            "blocker_code": clean_display_value(m.get("blocker_code"), label="—"),
            "fix": clean_display_value(m.get("recommended_fix"), label="—"),
            "runtime_ms": clean_display_value(m.get("runtime_ms"), status="not_applicable_smoke"),
        })
classify_df = pd.DataFrame(rows)
classify_df.to_csv(REPORTS_DIR / "classification_smoke_status_v34.csv", index=False)
classify_df


In [ ]:
# Classification smoke latency (only for smoke_passed rows)
ok = [r for r in classify_df.to_dict('records') if r['final_state'] == 'smoke_passed']
if ok:
    labels = [r['model_id'] for r in ok]
    # runtime_ms is a string; reparse from raw smoke matrix
    raw_runtimes = {m['model_id']: float(m.get('runtime_ms') or 0) for m in smoke['rows']}
    values = [raw_runtimes.get(l, 0)/1000.0 for l in labels]
    fig, ax = plt.subplots(figsize=(9, max(3, 0.4*len(labels))))
    ax.barh(labels, values, color="#3b82f6")
    ax.set_xlabel("smoke runtime (s)")
    ax.set_title("Classification smoke runtimes (process wall time, includes load)")
    for i, v in enumerate(values):
        ax.text(v + 0.05, i, f"{v:.2f}s", va="center")
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / "classification_smoke_runtime.png", dpi=130)
    plt.show()
else:
    print("No classification smoke_passed rows.")


## 7. Embedding / similarity — smoke only

Cosine similarity on a tiny image pair set. Real retrieval benchmark
requires GT retrieval pairs (not supplied).

In [ ]:
EMBED_MODELS = [
    "dinov2-base", "dinov2-small", "dinov2-large", "dinov2-giant",
    "clip-vit-base-patch32", "clip-vit-large-patch14",
    "siglip2-base-patch16-224", "siglip2-large-patch16-256",
    "siglip2-so400m-patch14-384", "siglip-base-patch16-224",
]
rows = []
for m in smoke["rows"]:
    if m["model_id"] in EMBED_MODELS:
        rows.append({
            "model_id": m["model_id"],
            "final_state": m["final_state"],
            "blocker_code": clean_display_value(m.get("blocker_code"), label="—"),
            "fix": clean_display_value(m.get("recommended_fix"), label="—"),
        })
embed_df = pd.DataFrame(rows)
embed_df.to_csv(REPORTS_DIR / "embedding_smoke_status_v34.csv", index=False)
embed_df


## 8. Closed-set object detection — real COCO val2017 400-image benchmark

Source: `reports/detection_leaderboard_400_v227_source.csv` — the v2.27
real benchmark on a 400-image, object-rich, balanced COCO val2017
subset. **This is a real benchmark, not a smoke test.**

In [ ]:
det_src = pd.read_csv(SOURCE_REPORTS / "detection_leaderboard_400_v227_source.csv")
det_src = det_src.sort_values("mAP50_95", ascending=False).reset_index(drop=True)

# Render any NaN cells as 'not collected'
det_view = det_src.copy()
for col in ("mAP50_95", "AP50", "AP75", "latency_ms_p50", "latency_ms_p95", "runtime_s"):
    if col in det_view.columns:
        det_view[col + "_display"] = det_view[col].apply(lambda v: render_nullable(v, status="not_collected"))

# Write canonical CSV/JSON to v34 reports
det_src.to_csv(REPORTS_DIR / "detection_leaderboard_400_v34.csv", index=False)
det_src.to_json(REPORTS_DIR / "detection_leaderboard_400_v34.json", orient="records", indent=2)
print(f"Detection rows: {len(det_src)}")
det_view[["rank","model_id","source_engine","family","n_images","mAP50_95_display","AP50_display","AP75_display","latency_ms_p50_display","status"]].head(20)


In [ ]:
# Plot 1: mAP50:95 bar
fig, ax = plt.subplots(figsize=(11, max(4, 0.32*len(det_src))))
ax.barh(det_src["model_id"][::-1], det_src["mAP50_95"][::-1], color="#3b82f6")
ax.set_xlabel("mAP50:95 (COCO val2017, 400 images)")
ax.set_title("Closed-set object detection — mAP50:95")
for i, v in enumerate(det_src["mAP50_95"][::-1]):
    if pd.notna(v):
        ax.text(v + 0.005, i, f"{v:.4f}", va="center", fontsize=8)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "detection_mAP50_95_by_model.png", dpi=130)
plt.show()


In [ ]:
# Plot 2: AP50 bar
fig, ax = plt.subplots(figsize=(11, max(4, 0.32*len(det_src))))
ax.barh(det_src["model_id"][::-1], det_src["AP50"][::-1], color="#16a34a")
ax.set_xlabel("AP50 (COCO val2017, 400 images)")
ax.set_title("Closed-set object detection — AP50")
for i, v in enumerate(det_src["AP50"][::-1]):
    if pd.notna(v):
        ax.text(v + 0.005, i, f"{v:.4f}", va="center", fontsize=8)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "detection_AP50_by_model.png", dpi=130)
plt.show()


In [ ]:
# Plot 3: latency p50 + FPS only for rows that have latency
lat = det_src.dropna(subset=["latency_ms_p50"]).sort_values("latency_ms_p50")
if len(lat) > 0:
    fig, ax = plt.subplots(figsize=(11, max(3, 0.4*len(lat))))
    ax.barh(lat["model_id"], lat["latency_ms_p50"], color="#a855f7")
    ax.set_xlabel("latency p50 (ms, lower is better)")
    ax.set_title("Closed-set object detection — latency p50")
    for i, v in enumerate(lat["latency_ms_p50"]):
        ax.text(v + 0.2, i, f"{v:.1f} ms", va="center", fontsize=8)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / "detection_latency_p50_by_model.png", dpi=130)
    plt.show()

    fps = 1000.0 / lat["latency_ms_p50"]
    fig, ax = plt.subplots(figsize=(11, max(3, 0.4*len(lat))))
    ax.barh(lat["model_id"], fps, color="#fbbf24")
    ax.set_xlabel("FPS (1000/p50)")
    ax.set_title("Closed-set object detection — FPS")
    for i, v in enumerate(fps):
        ax.text(v + 1, i, f"{v:.1f} FPS", va="center", fontsize=8)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / "detection_fps_by_model.png", dpi=130)
    plt.show()


In [ ]:
# Pareto: mAP50:95 vs latency
par = det_src.dropna(subset=["latency_ms_p50","mAP50_95"])
if len(par) > 1:
    fig, ax = plt.subplots(figsize=(9,6))
    colors = {"ultralytics":"#1f77b4","visionservex":"#ff7f0e","libreyolo":"#2ca02c","deimv2_sidecar":"#9467bd"}
    for engine, sub in par.groupby("source_engine"):
        ax.scatter(sub["latency_ms_p50"], sub["mAP50_95"],
                   s=70, label=engine, color=colors.get(engine, "gray"))
        for _, r in sub.iterrows():
            ax.annotate(r["model_id"], (r["latency_ms_p50"], r["mAP50_95"]),
                        xytext=(4,4), textcoords="offset points", fontsize=7)
    ax.set_xlabel("latency p50 (ms)")
    ax.set_ylabel("mAP50:95")
    ax.set_title("Closed-set detection Pareto: mAP vs latency")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / "detection_pareto_map_latency.png", dpi=130)
    plt.show()

    par_out = par[["model_id","source_engine","family","mAP50_95","latency_ms_p50"]].copy()
    par_out.to_csv(REPORTS_DIR / "detection_pareto_v34.csv", index=False)


In [ ]:
# Family-size curves
def parse_family_size(row):
    fam = row["family"]
    m = row["model_id"].lower()
    for tok in ("nano","small","medium","base","large","xlarge","2xlarge",
                "n","s","m","l","x"):
        if f"-{tok}" in m or f"_{tok}" in m or m.endswith(tok):
            return fam, tok
    return fam, "?"

sizes_order = {"nano":0,"n":0,"atto":-2,"femto":-1.5,"pico":-1,
               "small":1,"s":1,"medium":2,"m":2,"base":2.5,"b":2.5,
               "large":3,"l":3,"xlarge":4,"x":4,"2xlarge":5}

curves = []
for _, r in det_src.iterrows():
    fam, size = parse_family_size(r)
    curves.append({"family": fam, "size": size,
                   "size_rank": sizes_order.get(size, -10),
                   "model_id": r["model_id"],
                   "mAP50_95": r["mAP50_95"]})
curves_df = pd.DataFrame(curves)
curves_df.to_csv(REPORTS_DIR / "detection_family_size_summary_v34.csv", index=False)

fig, ax = plt.subplots(figsize=(10,6))
for fam, sub in curves_df.groupby("family"):
    sub = sub.sort_values("size_rank")
    ax.plot(sub["size"], sub["mAP50_95"], marker="o", label=fam, linewidth=2)
    for _, r in sub.iterrows():
        ax.annotate(r["model_id"], (r["size"], r["mAP50_95"]),
                    xytext=(4,4), textcoords="offset points", fontsize=7)
ax.set_xlabel("model size variant")
ax.set_ylabel("mAP50:95")
ax.set_title("Closed-set detection — family-size mAP curves")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / "detection_family_size_curves.png", dpi=130)
plt.show()


In [ ]:
# Detection status matrix
det_status = det_src[["model_id","source_engine","family","status"]].copy()
det_status["display_status"] = det_status["status"].apply(
    lambda s: "benchmark_passed" if s == "ok" else "expected_blocker"
)
det_status.to_csv(REPORTS_DIR / "detection_status_v34.csv", index=False)
det_status


## 9. Open-vocabulary detection — smoke

Closed-set detection benchmark above does not apply; open-vocab is a
different protocol (free-form text prompts, no fixed COCO classes).

In [ ]:
OV_MODELS = [
    "owlv2-base-patch16","owlv2-large-patch14",
    "owlvit-base-patch32","owlvit-large-patch14",
    "grounding-dino-tiny","grounding-dino-swin-t","grounding-dino-swin-b",
    "florence-2-base","florence-2-large",
]
rows = []
for m in smoke["rows"]:
    if m["model_id"] in OV_MODELS:
        rows.append({
            "model_id": m["model_id"], "final_state": m["final_state"],
            "blocker_code": clean_display_value(m.get("blocker_code"), label="—"),
            "fix": clean_display_value(m.get("recommended_fix"), label="—"),
        })
ov_df = pd.DataFrame(rows)
ov_df.to_csv(REPORTS_DIR / "open_vocab_smoke_status_v34.csv", index=False)
ov_df


## 10. Automatic instance segmentation — v2.27 COCO 400 + v34 status

Real Ultralytics auto-seg numbers come from v2.27 evidence. RF-DETR-Seg
schema is confirmed (`segments[i].mask` uint8 (H,W)); full mask AP
requires pycocotools and is reserved for the next benchmark pass.

In [ ]:
seg_src = json.loads((SOURCE_REPORTS / "segmentation_auto_instance_400_v227_source.json").read_text())
seg_rows = seg_src.get("rows", [])
seg_df = pd.DataFrame(seg_rows)

# Plot if available
if "mask_mAP50_95" in seg_df.columns and len(seg_df):
    seg_df_sorted = seg_df.sort_values("mask_mAP50_95", ascending=False)
    seg_df_sorted.to_csv(REPORTS_DIR / "automatic_segmentation_leaderboard_400_v34.csv", index=False)
    seg_df_sorted.to_json(REPORTS_DIR / "automatic_segmentation_leaderboard_400_v34.json", orient="records", indent=2)

    fig, ax = plt.subplots(figsize=(10, max(3, 0.5*len(seg_df_sorted))))
    ax.barh(seg_df_sorted["model_id"][::-1], seg_df_sorted["mask_mAP50_95"][::-1], color="#8b5cf6")
    ax.set_xlabel("mask mAP50:95")
    ax.set_title("Automatic instance segmentation — mask mAP50:95 (COCO val2017 400)")
    for i, v in enumerate(seg_df_sorted["mask_mAP50_95"][::-1]):
        ax.text(v + 0.005, i, f"{v:.4f}", va="center", fontsize=8)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / "automatic_segmentation_mask_mAP50_95.png", dpi=130)
    plt.show()

    if "mask_AP50" in seg_df_sorted.columns:
        fig, ax = plt.subplots(figsize=(10, max(3, 0.5*len(seg_df_sorted))))
        ax.barh(seg_df_sorted["model_id"][::-1], seg_df_sorted["mask_AP50"][::-1], color="#16a34a")
        ax.set_xlabel("mask AP50")
        ax.set_title("Automatic instance segmentation — mask AP50")
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / "automatic_segmentation_AP50.png", dpi=130)
        plt.show()

    if "latency_ms_p50" in seg_df_sorted.columns:
        lat = seg_df_sorted.dropna(subset=["latency_ms_p50"]).sort_values("latency_ms_p50")
        if len(lat) > 0:
            fig, ax = plt.subplots(figsize=(10, max(3, 0.5*len(lat))))
            ax.barh(lat["model_id"], lat["latency_ms_p50"], color="#a855f7")
            ax.set_xlabel("latency p50 (ms)")
            ax.set_title("Automatic segmentation — latency p50")
            plt.tight_layout()
            fig.savefig(PLOTS_DIR / "automatic_segmentation_latency.png", dpi=130)
            plt.show()

    if {"mask_mAP50_95","latency_ms_p50"}.issubset(seg_df_sorted.columns):
        par = seg_df_sorted.dropna(subset=["mask_mAP50_95","latency_ms_p50"])
        if len(par) > 1:
            fig, ax = plt.subplots(figsize=(8,6))
            ax.scatter(par["latency_ms_p50"], par["mask_mAP50_95"], s=70, color="#8b5cf6")
            for _, r in par.iterrows():
                ax.annotate(r["model_id"], (r["latency_ms_p50"], r["mask_mAP50_95"]),
                            xytext=(4,4), textcoords="offset points", fontsize=7)
            ax.set_xlabel("latency p50 (ms)")
            ax.set_ylabel("mask mAP50:95")
            ax.set_title("Auto segmentation Pareto")
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            fig.savefig(PLOTS_DIR / "automatic_segmentation_pareto.png", dpi=130)
            plt.show()
else:
    print("No mask_mAP50_95 column in source; only Ultralytics rows are benchmarked.")

seg_df


In [ ]:
# RF-DETR-Seg schema status (v34)
schema_probe = json.loads((SOURCE_REPORTS / "rfdetr_seg_schema_probe_v229.json").read_text())
schema_row = {
    "model": "rfdetr-seg-small",
    "schema_status": schema_probe.get("status"),
    "mask_format": schema_probe.get("mask_format"),
    "mask_field_path": schema_probe.get("mask_field_path"),
    "mask_dtype": schema_probe.get("mask_dtype"),
    "n_segments_observed": schema_probe.get("n_segments_observed"),
    "next_action": "implement COCO RLE conversion via pycocotools to enable mask AP",
}
print(json.dumps(schema_row, indent=2))


## 11. Promptable segmentation — smoke/eval with GT box prompts

Real protocol: GT bbox → mask prediction → IoU vs GT mask. The current
COCO smoke asset has GT polygons, so an IoU can be computed.

In [ ]:
ann_path = SMOKE_ASSETS / "coco_instance_sample.json"
images_dir = SMOKE_ASSETS

PROMPT_MODELS = ["sam2-hiera-tiny", "sam2.1-hiera-tiny"]
ps_rows = []

try:
    # Use the package CLI for stability
    out_path = REPORTS_DIR / "promptable_segmentation_smoke_eval_v34.json"
    cmd = [
        sys.executable, "-m", "visionservex",
        "benchmark-promptable-segmentation",
        "--dataset", str(ann_path),
        "--images-dir", str(images_dir),
        "--models", ",".join(PROMPT_MODELS),
        "--prompt-source", "gt-box",
        "--max-instances", "3",
        "--device", "cuda",
        "--format", "json",
        "--out", str(out_path),
        "--draw-dir", str(VISUALS_DIR / "promptable_seg_overlays"),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
    if out_path.exists():
        ps = json.loads(out_path.read_text())
    else:
        ps = json.loads(proc.stdout.strip())
    for r in ps.get("rows", []):
        ious = [inst.get("iou") for inst in r.get("rows", []) if isinstance(inst.get("iou"), (int, float))]
        mean_iou = sum(ious)/len(ious) if ious else None
        ps_rows.append({
            "model_id": r.get("model_id"),
            "status": r.get("status"),
            "total_instances": r.get("total_instances", 0),
            "mean_iou_display": clean_display_value(mean_iou, status="not_applicable_smoke"),
            "metric_status": r.get("metric_status", "not_applicable"),
        })
except Exception as exc:
    ps_rows.append({"model_id": "_error", "status": "expected_blocker",
                    "fix": str(exc)[:200]})

ps_df = pd.DataFrame(ps_rows)
ps_df.to_csv(REPORTS_DIR / "promptable_segmentation_leaderboard_400_v34.csv", index=False)
ps_df


In [ ]:
# Promptable segmentation IoU bar (if any computed)
if not ps_df.empty and 'mean_iou_display' in ps_df.columns:
    # Parse numeric IoU back
    def parse_iou(s):
        try:
            return float(s)
        except Exception:
            return None
    ious = [parse_iou(x) for x in ps_df["mean_iou_display"]]
    valid = [(m, i) for m, i in zip(ps_df["model_id"], ious) if i is not None]
    if valid:
        labels = [v[0] for v in valid]
        values = [v[1] for v in valid]
        fig, ax = plt.subplots(figsize=(8, max(2, 0.5*len(labels))))
        ax.barh(labels, values, color="#10b981")
        ax.set_xlabel("mean IoU (GT-box prompted, smoke set)")
        ax.set_title("Promptable segmentation — mean IoU")
        for i, v in enumerate(values):
            ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=8)
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / "promptable_segmentation_mean_iou.png", dpi=130)
        plt.show()
    else:
        print("No computable mean IoU in the smoke set (no GT mask polygons usable).")


## 12. Medical — smoke/demo status only

No medical NIfTI GT dataset is supplied; this section is **demo only**.
COCO smoke ≠ medical benchmark.

In [ ]:
MED_MODELS = ["medsam"]
rows = []
for m in smoke["rows"]:
    if m["model_id"] in MED_MODELS:
        rows.append({
            "model_id": m["model_id"], "final_state": m["final_state"],
            "blocker_code": clean_display_value(m.get("blocker_code"), label="—"),
            "fix": clean_display_value(m.get("recommended_fix"), label="—"),
            "note": "smoke/demo only; medical benchmark requires NIfTI + GT masks",
        })

med_df = pd.DataFrame(rows)
med_df.to_csv(REPORTS_DIR / "medical_smoke_status_v34.csv", index=False)
med_df


## 13. Agriculture — smoke/demo only

CropWeed / agriculture is **demo only**; no labelled crop/weed dataset is supplied.

In [ ]:
ag_df = pd.DataFrame([{
    "task": "agriculture",
    "status": "smoke/demo only",
    "evidence": str(SMOKE_ASSETS / "crop_weed_sample.jpg"),
    "note": "No labelled crop/weed dataset supplied; real benchmark requires explicit GT.",
}])
ag_df.to_csv(REPORTS_DIR / "agriculture_status_v34.csv", index=False)
ag_df


## 14. Aerial / OBB — expected_blocker

DOTA / VisDrone weights and labels are non-commercial / unverified.
This task is `dataset_required` until a permissive OBB dataset is added.

In [ ]:
aerial_df = pd.DataFrame([{
    "task": "aerial_obb", "final_state": "dataset_required",
    "blocker_code": "OBB_DATASET_NOT_AUDITED",
    "fix": "Supply a permissive OBB dataset (e.g. user-labelled GT)",
}])
aerial_df.to_csv(REPORTS_DIR / "aerial_obb_status_v34.csv", index=False)
aerial_df


## 15. Anomaly — smoke / expected_blocker

In [ ]:
anom_cmd = [sys.executable, "-m", "visionservex", "anomaly", "doctor",
             "--format", "json", "--out", str(REPORTS_DIR / "anomaly_doctor_v34.json")]
proc = subprocess.run(anom_cmd, capture_output=True, text=True, timeout=30)
try:
    anom = json.loads(proc.stdout.strip()) if proc.stdout.strip().startswith("{") else json.loads((REPORTS_DIR / "anomaly_doctor_v34.json").read_text())
except Exception:
    anom = {"status": "expected_blocker", "code": "ANOMALIB_REQUIRED"}

anom_row = {
    "task": "anomaly",
    "final_state": "expected_blocker" if anom.get("code") == "ANOMALIB_REQUIRED" else anom.get("status"),
    "blocker_code": anom.get("code", "ANOMALIB_REQUIRED"),
    "fix": "pip install 'visionservex[anomaly]'",
    "anomalib_installed": anom.get("anomalib_installed", False),
}
anom_df = pd.DataFrame([anom_row])
anom_df.to_csv(REPORTS_DIR / "anomaly_status_v34.csv", index=False)
anom_df


## 16. Surveillance / Video tracking — smoke / expected_blocker

In [ ]:
def _run_smoke(args):
    proc = subprocess.run(args, capture_output=True, text=True, timeout=60)
    try:
        return json.loads(proc.stdout.strip())
    except Exception:
        return {"status": "expected_blocker", "code": "STRUCTURED_PAYLOAD_MISSING"}

tracker_rows = []
for t in ("bytetrack", "oc-sort"):
    out = REPORTS_DIR / f"{t.replace('-', '_')}_smoke_v34.json"
    cmd = [sys.executable, "-m", "visionservex", "video-search", "tracker-smoke",
           "--tracker", t, "--format", "json", "--out", str(out)]
    p = _run_smoke(cmd)
    tracker_rows.append({
        "task": "tracking", "tracker": t,
        "final_state": "expected_blocker" if p.get("status") == "expected_blocker" else "smoke_passed",
        "blocker_code": p.get("code", "—"),
        "fix": p.get("install", "—"),
    })

# OSNet (reid)
osnet_out = REPORTS_DIR / "osnet_smoke_v34.json"
osnet_cmd = [sys.executable, "-m", "visionservex", "video-search", "reid-smoke",
             "--image", str(SMOKE_ASSETS / "coco_person_car.jpg"),
             "--reid", "osnet", "--format", "json", "--out", str(osnet_out)]
osnet = _run_smoke(osnet_cmd)
tracker_rows.append({
    "task": "reid", "tracker": "osnet",
    "final_state": "expected_blocker" if osnet.get("status") == "expected_blocker" else "smoke_passed",
    "blocker_code": osnet.get("code", "—"),
    "fix": osnet.get("install", "—"),
})

# Video annotate dry-run
vid_out = REPORTS_DIR / "video_annotate_smoke_v34.json"
vid_cmd = [sys.executable, "-m", "visionservex", "annotate", "video",
           "--video", str(SMOKE_ASSETS / "tracking_sample.mp4"),
           "--model", "dfine-s-o365-coco",
           "--out", str(VISUALS_DIR / "video_annotated_v34.mp4")]
proc = subprocess.run(vid_cmd, capture_output=True, text=True, timeout=180)
tracker_rows.append({
    "task": "video_annotate", "tracker": "dfine-s-o365-coco",
    "final_state": "smoke_passed" if proc.returncode == 0 else "expected_blocker",
    "blocker_code": "—" if proc.returncode == 0 else "VIDEO_ANNOTATE_FAILED",
    "fix": "—",
})

vid_df = pd.DataFrame(tracker_rows)
vid_df.to_csv(REPORTS_DIR / "surveillance_smoke_status_v34.csv", index=False)
vid_df


## 17. Remaining blockers (consolidated)

In [ ]:
blocker_rows = []
for r in smoke["rows"]:
    if r["final_state"] in (
        "expected_blocker", "dependency_required", "download_failed_retryable",
        "license_blocked", "manual_checkpoint_required", "dataset_required",
        "upstream_unavailable",
    ):
        blocker_rows.append({
            "model_id": r["model_id"],
            "task": r["task"],
            "final_state": r["final_state"],
            "blocker_code": r["blocker_code"] or "—",
            "fix": r.get("recommended_fix") or "—",
            "package_bug": bool(r.get("package_bug", False)),
            "external_blocker": bool(r.get("external_blocker", True)),
        })

# Add tracker / anomaly blockers from sections above
for tr in tracker_rows:
    if tr["final_state"] in ("expected_blocker", "dependency_required"):
        blocker_rows.append({
            "model_id": tr["tracker"], "task": tr["task"],
            "final_state": tr["final_state"], "blocker_code": tr["blocker_code"],
            "fix": tr["fix"], "package_bug": False, "external_blocker": True,
        })

# Add RT-DETRv4 manual-checkpoint variants
rtv4 = json.loads((SOURCE_REPORTS / "rtdetrv4_checkpoint_audit_v230.json").read_text())
for r in rtv4.get("rows", []):
    blocker_rows.append({
        "model_id": r["model_id"], "task": "detect",
        "final_state": r["final_state"], "blocker_code": r["blocker_code"],
        "fix": r["gdown_command"], "package_bug": False, "external_blocker": True,
    })

blocker_df = pd.DataFrame(blocker_rows)
blocker_df.to_csv(REPORTS_DIR / "blocker_summary_v34.csv", index=False)
print(f"Total blockers: {len(blocker_df)}")
blocker_df


In [ ]:
# Blocker breakdown plot
counts = blocker_df["blocker_code"].value_counts() if "blocker_code" in blocker_df.columns else pd.Series(dtype=int)
if len(counts):
    fig, ax = plt.subplots(figsize=(10, max(3, 0.4*len(counts))))
    counts.plot(kind="barh", ax=ax, color="#dc2626")
    ax.set_title("v34 — Blocker code breakdown (all tasks)")
    ax.set_xlabel("count")
    for i, v in enumerate(counts.values):
        ax.text(v + 0.05, i, str(v), va="center")
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / "blocker_breakdown.png", dpi=130)
    plt.show()


## 18. Task coverage heatmap

In [ ]:
tasks = sorted({r["task"] for r in smoke["rows"]})
states = ALLOWED_FINAL_STATES
mat = np.zeros((len(tasks), len(states)), dtype=int)
for r in smoke["rows"]:
    if r["final_state"] in states:
        mat[tasks.index(r["task"]), states.index(r["final_state"])] += 1

fig, ax = plt.subplots(figsize=(13, max(4, 0.6*len(tasks))))
im = ax.imshow(mat, aspect="auto", cmap="Blues")
ax.set_xticks(range(len(states)), labels=states, rotation=45, ha="right")
ax.set_yticks(range(len(tasks)), labels=tasks)
for i in range(len(tasks)):
    for j in range(len(states)):
        if mat[i,j] > 0:
            ax.text(j, i, mat[i,j], ha="center", va="center",
                    color="white" if mat[i,j] > mat.max()/2 else "black", fontsize=8)
ax.set_title("v34 — Task × final-state coverage (core models)")
fig.colorbar(im, ax=ax, label="model count")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "task_coverage_heatmap.png", dpi=130)
plt.show()


## 19. Final task-specific winner summary

There is **no overall winner** across tasks. Each task has its own
metric and its own benchmark dataset; cross-task comparisons are
scientifically meaningless.

In [ ]:
winners = {}

# Detection (real benchmark)
if "mAP50_95" in det_src.columns:
    top_map = det_src.dropna(subset=["mAP50_95"]).iloc[0]
    winners["detection_best_mAP50_95"] = {
        "model_id": top_map["model_id"], "source_engine": top_map["source_engine"],
        "mAP50_95": float(top_map["mAP50_95"]),
        "AP50": float(top_map["AP50"]) if not pd.isna(top_map["AP50"]) else None,
        "dataset": "COCO val2017, 400-image object-rich subset",
    }
if "latency_ms_p50" in det_src.columns:
    lat = det_src.dropna(subset=["latency_ms_p50"]).sort_values("latency_ms_p50")
    if len(lat):
        fastest = lat.iloc[0]
        winners["detection_fastest_p50"] = {
            "model_id": fastest["model_id"], "latency_ms_p50": float(fastest["latency_ms_p50"]),
            "mAP50_95": float(fastest["mAP50_95"]) if not pd.isna(fastest["mAP50_95"]) else None,
        }

# Auto segmentation
if "mask_mAP50_95" in seg_df.columns and len(seg_df):
    seg_top = seg_df.sort_values("mask_mAP50_95", ascending=False).iloc[0]
    winners["automatic_segmentation_best_mask_mAP50_95"] = {
        "model_id": seg_top["model_id"], "mask_mAP50_95": float(seg_top["mask_mAP50_95"]),
        "dataset": "COCO val2017 400 + Ultralytics yolo*-seg only (RF-DETR-Seg AP pending pycocotools)",
    }

# Promptable segmentation
if not ps_df.empty:
    winners["promptable_segmentation_status"] = {
        "models_attempted": ps_df["model_id"].tolist(),
        "note": "Smoke/eval with GT box prompts. Full COCO 400 promptable benchmark deferred.",
    }

# Classification & embedding: status-only because no labelled dataset
winners["classification_status"] = "smoke_only (no ImageNet labels supplied)"
winners["embedding_status"] = "smoke_only (no retrieval GT supplied)"

# Domain
winners["medical_status"] = "smoke_only (no NIfTI GT)"
winners["agriculture_status"] = "smoke_only (no crop/weed GT)"
winners["aerial_obb_status"] = "dataset_required"
winners["anomaly_status"] = "expected_blocker (ANOMALIB_REQUIRED)"
winners["surveillance_status"] = "expected_blocker (BYTETRACK/OCSORT/TORCHREID required)"

(REPORTS_DIR / "final_task_winner_summary_v34.json").write_text(json.dumps(winners, indent=2))

# Markdown version
lines = ["# VisionServeX v34 — Final task-specific winner summary", ""]
lines.append(f"VisionServeX version : `{VSX_VERSION}`")
lines.append(f"Notebook version     : `{NOTEBOOK_VERSION}`")
lines.append("")
lines.append("## Detection (real COCO val2017 400 benchmark)")
if "detection_best_mAP50_95" in winners:
    w = winners["detection_best_mAP50_95"]
    lines.append(f"- **Best mAP50:95**: `{w['model_id']}` ({w['source_engine']}) = `{w['mAP50_95']:.4f}` (AP50=`{w['AP50']:.4f}` if available)")
if "detection_fastest_p50" in winners:
    w = winners["detection_fastest_p50"]
    lines.append(f"- **Fastest p50**: `{w['model_id']}` at `{w['latency_ms_p50']:.2f} ms` (mAP=`{w['mAP50_95']:.4f}`)")
lines.append("")
lines.append("## Automatic instance segmentation (COCO val2017 400)")
if "automatic_segmentation_best_mask_mAP50_95" in winners:
    w = winners["automatic_segmentation_best_mask_mAP50_95"]
    lines.append(f"- **Best mask mAP50:95**: `{w['model_id']}` = `{w['mask_mAP50_95']:.4f}`")
    lines.append(f"- Scope: Ultralytics yolo*-seg only — RF-DETR-Seg AP pending pycocotools RLE.")
lines.append("")
lines.append("## Promptable segmentation")
if "promptable_segmentation_status" in winners:
    lines.append(f"- Models attempted: `{winners['promptable_segmentation_status']['models_attempted']}`")
    lines.append(f"- {winners['promptable_segmentation_status']['note']}")
lines.append("")
lines.append("## Other tasks")
for k in ("classification_status","embedding_status","medical_status","agriculture_status",
          "aerial_obb_status","anomaly_status","surveillance_status"):
    if k in winners:
        lines.append(f"- **{k}**: {winners[k]}")

(REPORTS_DIR / "final_task_winner_summary_v34.md").write_text("\n".join(lines))
print("\n".join(lines))


## 20. Scientific limitations (read before citing)

1. **Detection benchmark**: 400-image subset of COCO val2017 (object-rich,
   balanced). Smaller than the full 5000-image val set, so absolute
   numbers are not directly comparable to published papers. Relative
   ordering is stable.
2. **Automatic segmentation**: Only Ultralytics yolo*-seg models have
   real mask-AP rows. VisionServeX rfdetr-seg-* models have their schema
   confirmed but mask AP requires pycocotools — pending.
3. **Promptable segmentation**: smoke/eval set (2 images, ≤3 instances).
   Full COCO 400 box-prompted SAM benchmark is the next pass.
4. **Classification / embedding / domain (medical / agriculture)**:
   no labelled dataset supplied — smoke / demo status only.
5. **No cross-task comparisons.** Detection mAP and segmentation mask AP
   are not directly comparable; promptable IoU is not directly comparable
   to automatic AP.

## 21. Final audit report

In [ ]:
def _safe_count(field: str) -> int:
    return sum(1 for r in smoke["rows"] if r["final_state"] == field)

audit = {
    "visionservex_version": VSX_VERSION,
    "notebook_version": NOTEBOOK_VERSION,
    "package_smoke": {
        "total": len(smoke["rows"]),
        "smoke_passed": _safe_count("smoke_passed"),
        "benchmark_passed": _safe_count("benchmark_passed"),
        "expected_blocker": _safe_count("expected_blocker"),
        "dependency_required": _safe_count("dependency_required"),
        "download_failed_retryable": _safe_count("download_failed_retryable"),
        "manual_checkpoint_required": _safe_count("manual_checkpoint_required"),
        "license_blocked": _safe_count("license_blocked"),
        "failed_runtime": _safe_count("failed_runtime"),
        "unclassified": _safe_count("unclassified"),
        "package_bug_remaining": sum(1 for r in smoke["rows"] if r.get("package_bug")),
    },
    "detection": {
        "dataset": "COCO val2017, 400 images, object-rich balanced subset",
        "models_benchmarked": int(len(det_src)),
        "best_mAP50_95": winners.get("detection_best_mAP50_95"),
        "fastest_p50": winners.get("detection_fastest_p50"),
        "evidence": "reports/detection_leaderboard_400_v34.csv",
    },
    "automatic_segmentation": {
        "dataset": "COCO val2017, 400 images, mask GT",
        "models_benchmarked": int(len(seg_df)),
        "best_mask_mAP50_95": winners.get("automatic_segmentation_best_mask_mAP50_95"),
        "scope": "Ultralytics yolo*-seg only; RF-DETR-Seg AP pending pycocotools",
        "evidence": "reports/automatic_segmentation_leaderboard_400_v34.csv",
    },
    "promptable_segmentation": {
        "models_attempted": PROMPT_MODELS,
        "scope": "smoke/eval set; full COCO 400 promptable benchmark deferred",
        "evidence": "reports/promptable_segmentation_leaderboard_400_v34.csv",
    },
    "domain": {
        "medical": "smoke/demo only — no NIfTI GT",
        "agriculture": "smoke/demo only — no crop/weed GT",
        "aerial_obb": "dataset_required",
        "anomaly": "expected_blocker (ANOMALIB_REQUIRED)",
        "surveillance": "expected_blocker (BYTETRACK / OCSORT / TORCHREID)",
    },
    "libreyolo": {
        "version": ly_doctor.get("libreyolo_version"),
        "n_weights_discovered": ly_models.get("n_weights"),
        "n_default_safe": n_safe,
        "n_blocked": n_blocked,
    },
    "v3_ready": False,
    "v3_blockers": [
        "RT-DETRv4 manual checkpoints",
        "RF-DETR-Seg pycocotools mask-AP integration",
        "Full COCO 400 promptable segmentation benchmark",
        "Labelled ImageNet-style classification dataset",
        "Labelled retrieval dataset for embedding benchmark",
        "Permissive aerial/OBB dataset",
        "anomalib install",
        "natten install",
        "transformers<5.0 sidecar for Florence-2",
    ],
}
(REPORTS_DIR / "final_audit_report_v34.json").write_text(json.dumps(audit, indent=2))

md_lines = ["# VisionServeX v34 — Final audit report", ""]
md_lines.append(f"VisionServeX `{audit['visionservex_version']}` / notebook `{audit['notebook_version']}`")
md_lines.append("")
md_lines.append("## Package smoke matrix")
for k, v in audit["package_smoke"].items():
    md_lines.append(f"- {k}: {v}")
md_lines.append("")
md_lines.append("## Detection")
det_w = audit["detection"]
md_lines.append(f"- Dataset: {det_w['dataset']}")
md_lines.append(f"- Models benchmarked: {det_w['models_benchmarked']}")
if det_w["best_mAP50_95"]:
    md_lines.append(f"- Best mAP50:95: `{det_w['best_mAP50_95']['model_id']}` = `{det_w['best_mAP50_95']['mAP50_95']:.4f}`")
md_lines.append("")
md_lines.append("## Automatic segmentation")
seg_w = audit["automatic_segmentation"]
md_lines.append(f"- Dataset: {seg_w['dataset']}")
md_lines.append(f"- Models benchmarked: {seg_w['models_benchmarked']}")
if seg_w.get("best_mask_mAP50_95"):
    md_lines.append(f"- Best mask mAP50:95: `{seg_w['best_mask_mAP50_95']['model_id']}` = `{seg_w['best_mask_mAP50_95']['mask_mAP50_95']:.4f}`")
md_lines.append(f"- Scope: {seg_w['scope']}")
md_lines.append("")
md_lines.append("## Promptable segmentation")
ps_w = audit["promptable_segmentation"]
md_lines.append(f"- Models attempted: `{ps_w['models_attempted']}`")
md_lines.append(f"- {ps_w['scope']}")
md_lines.append("")
md_lines.append("## v3 readiness")
md_lines.append(f"- v3_ready: **{audit['v3_ready']}**")
md_lines.append(f"- Blockers:")
for b in audit["v3_blockers"]:
    md_lines.append(f"  - {b}")

(REPORTS_DIR / "final_audit_report_v34.md").write_text("\n".join(md_lines))
print("\n".join(md_lines[:30]))


## 22. Reproducibility manifest

All artifacts written this run:

In [ ]:
manifest = sorted(p.relative_to(OUT_ROOT).as_posix() for p in OUT_ROOT.rglob("*") if p.is_file())
print(f"Total artifacts: {len(manifest)}")
for f in manifest:
    print(f"  {f}")
(REPORTS_DIR / "manifest_v34.txt").write_text("\n".join(manifest))
